# Machine Learning Workflow

This is the final session in our machine learning series. This session
shifts attention to the workflow itself, showing what an end-to-end
workflow actually looks like from problem definition through to
deployment.

We will use the Titanic dataset again, for familiarity, but the goal
here is not the model, it is the process. We will learn about feature
engineering, cross-validation, pipelines, and hyperparameter tuning.

## Slides

Use the left ⬅️ and right ➡️ arrow keys to navigate through the slides
below. To view in a separate tab/window,
<a href="slides.html" target="_blank">follow this link</a>.

## The Full Workflow

A complete machine learning project has six stages:

1.  **Problem definition** - what are you actually trying to predict,
    and why?
2.  **Data** - what do you have, what’s missing, what needs fixing?
3.  **Feature engineering** - transforming raw data into useful model
    inputs
4.  **Modelling** - selecting and training a model
5.  **Validation** - honestly evaluating whether it works
6.  **Deployment** - putting the model to use and keeping it working

Many tutorials focus almost entirely on modelling. In practice, data
preparation and feature engineering account for 60–80% of project time.
Model training is the smallest part of the work. The sessions so far
have covered data preparation and modelling in isolation. This session
puts them together in the right order, with the right safeguards.

## Problem Definition

A baseline estimate is essential before modelling. If 62% of passengers
did not survive, a model that always predicts “did not survive” achieves
62% accuracy while being completely useless. Any model we build must
outperform this to demonstrate it has learned something.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)

## Load and Explore

In [2]:
df = sns.load_dataset('titanic')

print(f"Shape: {df.shape}")
print(f"\nSurvival rate: {df['survived'].mean():.1%}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

Shape: (891, 15)

Survival rate: 38.4%

Missing values:
age            177
embarked         2
deck           688
embark_town      2
dtype: int64

Age has a meaningful number of missing values. Deck is almost entirely
missing and not worth using. Embarked has two missing values, which is
negligible but needs handling.

## Feature Engineering

In previous sessions, features were prepared with a function applied to
the train and test data. Here we will be more deliberate, to demonstrate
why each transformation is carried out and what information each feature
carries.

In [3]:
# select the columns we want to work with
df = df[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'alone', 'embark_town']].copy()

# encode sex as a binary integer
df['sex'] = (df['sex'] == 'female').astype(int)

# create dummy variables for embark town
df['embark_town'] = df['embark_town'].str.lower()
df = pd.get_dummies(df, columns=['embark_town'], prefix="embark")

df.head()

Age and fare still have missing values. These will be handled inside a
Pipeline rather than here, the reason for this is explained below.

## Train/Test Split

The split comes before any imputation or scaling. This is deliberate.
Preprocessing steps must be fit only on training data, never on the full
dataset.

In [4]:
X = df.drop(columns='survived')
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set:     {X_test.shape[0]} rows")

Training set: 712 rows
Test set:     179 rows

## Baseline Model

In [5]:
dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(X_train, y_train)
baseline_accuracy = accuracy_score(y_test, dummy_clf.predict(X_test))
print(f"Baseline accuracy: {baseline_accuracy:.1%}")

Baseline accuracy: 61.5%

## Building a Pipeline

In [6]:
pipe = Pipeline([
    # fill missing age and fare with training medians
    ('imputer', SimpleImputer(strategy='median')),
    # scale all features to zero mean, unit variance
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42))
])

## Cross-Validation

In [7]:
cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')

print(f"Cross-validated accuracy scores: {cv_scores.round(2)}")
print(f"Mean: {cv_scores.mean():.2f}  (+/- {cv_scores.std():.2f})")

Cross-validated accuracy scores: [0.79 0.71 0.82 0.85 0.81]
Mean: 0.80  (+/- 0.05)

## Hyperparameter Tuning

In [8]:
param_grid = {
    'model__n_estimators': [100, 300],
    'model__max_depth': [3, 5, None],
    'model__min_samples_leaf': [1, 5],
}

grid_search = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

print(f"Best parameters:  {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.2f}")

Best parameters:  {'model__max_depth': 3, 'model__min_samples_leaf': 1, 'model__n_estimators': 100}
Best CV accuracy: 0.84

## Final Evaluation

In [9]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred)

print(f"Baseline accuracy: {baseline_accuracy:.1%}")
print(f"Final model accuracy: {final_accuracy:.1%}")
print(f"Improvement over baseline: {final_accuracy - baseline_accuracy:+.1%}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=['Did not survive', 'Survived']))

Baseline accuracy: 61.5%
Final model accuracy: 79.3%
Improvement over baseline: +17.9%

Classification report:
                 precision    recall  f1-score   support

Did not survive       0.77      0.94      0.85       110
       Survived       0.85      0.57      0.68        69

       accuracy                           0.79       179
      macro avg       0.81      0.75      0.76       179
   weighted avg       0.80      0.79      0.78       179


## Deployment

## Where Next?

## Summary